# 🧠 تحليل `FeatureExtractor` — Sprint 7، الخطوة 1

---

## 🎯 TL;DR

> **`FeatureExtractor`** يحوّل **سلسلة شموع** → **مصفوفة ميزات (X)** + **أهداف تصنيفية (y)**. **5 ميزات** لكل عينة: `price_return`, `sma_diff`, `candle_range`, `rsi_norm`, `macd_norm`. **الهدف**: هل الشمعة التالية صاعدة (1) أم لا (0)؟

---

## 📊 المخطط المعماري

```mermaid
graph TD
    A[list Candle] --> B[FeatureExtractor]
    B --> C[RSI series]
    B --> D[SMA series]
    B --> E[MACD series]
    C --> F[Loop i=30 to n-2]
    D --> F
    E --> F
    F --> G{None in indicators?}
    G -->|Yes| H[skip]
    G -->|No| I[build 5 features]
    I --> J[target = next > curr?]
    J --> K[X.append + y.append]
    K --> F
    F --> L[return X, y]

    style B fill:#bfb,stroke:#333,stroke-width:3px
    style L fill:#bdf,stroke:#333,stroke-width:2px
```

---

## 🔬 التشريح

### 1️⃣ `__init__` — Composition

```python
self._rsi = RSI(period=rsi_period)
self._sma = SMA(period=sma_period)
self._macd = MACD(12, 26, 9)
```

**🎯 إعادة استخدام** مؤشرات Sprint 3 — لا تكرار.

---

### 2️⃣ `extract_features` — التدريب

#### الحد الأدنى للبيانات

```python
min_required = 35
if len(candles) < min_required:
    return [], []
```

**🎯 لماذا 35؟**
- **MACD(26)** يحتاج 26 شمعة.
- **Signal(9)** يحتاج 26+9−1 = **34 شمعة**.
- **+1** للهدف (الشمعة التالية) = **35**.

**Fail Soft:** إرجاع `[], []` بدل exception.

#### ترتيب الشموع

```python
candles_sorted = sorted(candles, key=lambda c: c.timestamp)
```

**🎯 لا نثق بالمدخل** — نفس نمط `SMA`/`EMA`.

#### حساب السلاسل

```python
rsi_series = self._rsi.calculate(candles_sorted)
sma_series = self._sma.calculate(candles_sorted)
macd_series = self._macd.calculate(candles_sorted)
```

**🎯 حساب مرة واحدة** لكل السلسلة — وليس per-candle (كفاءة).

#### Loop — من 30 إلى n−2

```python
start_idx = 30
for i in range(start_idx, len(candles_sorted) - 1):
```

| الحد | السبب |
|------|-------|
| `start_idx = 30` | MACD signal يكتمل عند 34، لكن 30 يعطي هامشاً (يتم فحص `None` بعدها) |
| `len - 1` | نحتاج `next_c = candles[i+1]` — فلا نصل للأخيرة |

**🎯 تريكة `len - 1`:** منع `IndexError` عند `next_c`.

#### الميزات الخمس

| # | الميزة | الصيغة | المعنى |
|---|--------|--------|--------|
| 1 | `price_return` | `(close - prev_close) / prev_close` | عائد لحظي |
| 2 | `sma_diff` | `(close - sma) / sma` | انحراف عن المتوسط |
| 3 | `candle_range` | `(high - low) / close` | تقلب الشمعة |
| 4 | `rsi_norm` | `rsi / 100` | RSI مطبع 0-1 |
| 5 | `macd_norm` | `macd` | خط MACD |

**🎯 تريكة التطبيع:**
- `rsi_norm` → `[0, 1]` بدل `[0, 100]` — **يساعد النموذج** (gradient descent أسرع).
- `price_return` و `sma_diff` و `candle_range` → **نسب مئوية** (float صغيرة) — **Stationary**.
- `macd_norm` → **بدون تطبيع** (المدى غير معروف مسبقاً).

#### الهدف (Target)

```python
target = 1 if next_c.close > curr_c.close else 0
```

| الحالة | القيمة |
|--------|--------|
| `next > curr` | `1` (صعود) |
| `next <= curr` | `0` (هبوط أو ثبات) |

**🎯 `<=` وليس `<`:** الثبات يُصنّف هبوطاً — **Binary Classification** بسيط.

**🎯 تريكة منع Lookahead:** `target` يستخدم `next_c` (المستقبل) — **صحيح للتدريب** فقط. في `extract_latest_features`، لا نستخدم `next_c`.

---

### 3️⃣ `extract_latest_features` — التنبؤ الحي

```python
if len(candles) < 35:
    return None
```

**🎯 `None` بدل `[]`:** يوضح "لا يمكن التنبؤ" — لا لبس مع "ميزات فارغة".

```python
curr_c = candles_sorted[-1]
prev_c = candles_sorted[-2]
```

**لا `next_c`** — لأننا نتنبأ بالمستقبل، لا نعرفه.

**نفس الـ 5 ميزات** — **اتساق تام** بين التدريب والتنبؤ (Training-Serving Skew Prevention).

---

## 📊 مقارنة `extract_features` vs `extract_latest_features`

| البعد | `extract_features` | `extract_latest_features` |
|-------|-------------------|---------------------------|
| **الغرض** | تدريب | تنبؤ حي |
| **المخرج** | `X, y` (كثير) | `list[float]` (واحد) |
| **يستخدم `next_c`** | ✅ (target) | ❌ |
| **Loop** | ✅ | ❌ |
| **عند نقص البيانات** | `[], []` | `None` |

---

## 🧪 التحقق (وصف)

**السيناريو:** 45 شمعة صاعدة تدريجياً.

| المقياس | القيمة |
|---------|--------|
| Samples | ~14 (45 − 30 − 1) |
| Features per Sample | 5 |
| Sample Target | `1` (لأن الصعود مستمر) |

**🎯 جوهر الاختبار:** إثبات أن الميزات **أرقام float** والهدف **0 أو 1**.

---

## ⚠️ ملاحظات على التصميم

| النقطة | التأثير |
|--------|---------|
| `start_idx = 30` ثابت | لا يتكيف مع `rsi_period`/`sma_period` مختلف |
| `None` check بعد `calculate` | صحيح — لكن قد يفوت صفوفاً |
| `macd_norm` بدون تطبيع | مدى غير متوقع — يؤثر على النموذج |
| لا `volume` feature | يمكن إضافته |
| لا `hour`/`day_of_week` | للـ intraday — غير مهم لـ `1day` |
| `min_required = 35` ثابت | يجب أن يكون `macd_slow + signal - 1 + 1` ديناميكياً |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| **Stationarity** | Returns بدل raw prices |
| **Normalization** | RSI/100 |
| **Composition** | RSI + SMA + MACD |
| **Lookahead Prevention** | `next_c` فقط للـ target |
| **Training-Serving Consistency** | نفس الـ 5 features |
| **Fail Soft** | `[], []` / `None` |
| **Defensive Sorting** | `sorted()` |
| **Efficiency** | حساب المؤشرات مرة واحدة |

---

## 🗺️ Sprint 7

```mermaid
graph LR
    A["ml/features.py 🆕"] --> B["ml/models/base.py + baseline.py"]
    B --> C["services/forecast_service.py"]
    C --> D["CLI: forex forecast"]
    D --> E["tests + docs"]

    style A fill:#bfb,stroke:#333,stroke-width:3px
    style B fill:#ddd,stroke:#333
```

In [1]:
from datetime import datetime, timedelta
from decimal import Decimal
from core.candle import Candle
from ml.features import FeatureExtractor

# إنشاء 45 شمعة تجريبية
base = datetime.now() - timedelta(days=45)
candles = [
    Candle(
        pair='EUR/USD', interval='1day',
        timestamp=base + timedelta(days=i),
        open=Decimal('1.0800') + Decimal(str(i * 0.0005)),
        high=Decimal('1.0850') + Decimal(str(i * 0.0005)),
        low=Decimal('1.0780') + Decimal(str(i * 0.0005)),
        close=Decimal('1.0820') + Decimal(str(i * 0.0005)),
    )
    for i in range(45)
]

extractor = FeatureExtractor()
X, y = extractor.extract_features(candles)
print(f'Samples Extracted: {len(X)}')
print(f'Features per Sample: {len(X[0]) if X else 0}')
print(f'Sample Feature Vector: {X[0] if X else None}')
print(f'Sample Target (0 or 1): {y[0] if y else None}')

Samples Extracted: 14
Features per Sample: 5
Sample Feature Vector: [0.00045599635202918376, 0.00434882124055848, 0.006381039197812215, 1.0, 0.0035]
Sample Target (0 or 1): 1


# 🤖 تحليل `BaseForecaster` + `LogisticForecaster` — Sprint 7، الخطوة 2

---

## 🎯 TL;DR

> **`BaseForecaster` (ABC)**: عقد بـ 3 دوال — `name` (property), `train(X, y)`, `predict_proba(features)` + `predict()` افتراضية. **`LogisticForecaster`**: تنفيذ بـ `sklearn` مع **Time-Series Split 80/20** (بدون Shuffle) و**إرجاع الاحتمالية** بدل التصنيف الثنائي.

---

## 📊 المخطط المعماري

```mermaid
graph TD
    A[BaseForecaster ABC] --> B[name property]
    A --> C[train]
    A --> D[predict_proba]
    A --> E[predict - concrete]
    F[LogisticForecaster] -.implements.-> A
    F --> G[sklearn LogisticRegression]
    F --> H[accuracy_score]
    E --> I{proba >= 0.5?}
    I -->|Yes| J[return 1 - UP]
    I -->|No| K[return 0 - DOWN]

    style A fill:#bbf,stroke:#333,stroke-width:3px
    style F fill:#bfb,stroke:#333,stroke-width:2px
    style E fill:#f9f,stroke:#333
```

---

## 🔬 التشريح

### 1️⃣ `BaseForecaster` — العقد (3 abstract + 1 concrete)

| الدالة | النوع | الدور |
|--------|-------|-------|
| `name` | `@property` abstract | اسم النموذج |
| `train(X, y)` | abstract | تدريب + إرجاع metrics |
| `predict_proba(features)` | abstract | احتمالية الصعود |
| **`predict(features)`** | **concrete** | تصنيف بناءً على `proba >= 0.5` |

**🎯 تريكة `predict` concrete:**

```python
def predict(self, features):
    return 1 if self.predict_proba(features) >= 0.50 else 0
```

**لماذا؟** كل نموذج ينتج **احتمالية** — `predict` تحويل ميكانيكي. تجنب تكرارها في كل نموذج.

**🎯 `>= 0.50` وليس `> 0.50`:** الثبات يُصنّف صعوداً — **قرار تصميمي** (conservative).

---

### 2️⃣ `LogisticForecaster` — التنفيذ

#### `__init__`

```python
self.model = LogisticRegression(C=c_param, max_iter=max_iter, random_state=42)
self.is_trained = False
self.last_accuracy = 0.0
```

| Parameter | Default | الدور |
|-----------|---------|-------|
| `C` | `1.0` | **Regularization inverse** — أصغر = تنظيم أقوى |
| `max_iter` | `200` | حد أقصى للتكرارات |
| `random_state` | `42` | **Reproducibility** |

**🎯 `is_trained` flag:** يمنع `predict_proba` قبل التدريب — **Fail Early**.

---

#### `train` — التدريب

```mermaid
graph TD
    A[X, y] --> B{n >= 20?}
    B -->|No| C[ValidationError]
    B -->|Yes| D[split_idx = int(n*0.8)]
    D --> E[X_train, X_test = X[:split], X[split:]]
    E --> F[model.fit X_train y_train]
    F --> G[y_pred = model.predict X_test]
    G --> H[last_accuracy = accuracy y_test y_pred]
    H --> I[return metrics dict]

    style B fill:#f9f,stroke:#333
    style F fill:#bfb,stroke:#333,stroke-width:2px
    style H fill:#bdf,stroke:#333
```

**الخطوات:**

| # | الخطوة | السبب |
|---|--------|-------|
| 1 | `if n < 20: raise` | **Fail Early** — بيانات غير كافية |
| 2 | `split_idx = int(n * 0.8)` | **Time-Series Split** 80/20 |
| 3 | `X[:split]` / `X[split:]` | **بدون Shuffle** — Train قديم، Test حديث |
| 4 | `model.fit` | تدريب على Train فقط |
| 5 | `predict(X_test)` | تقييم على Test **out-of-sample** |
| 6 | `accuracy_score` | دقة على البيانات **غير المرئية** |

**🎯 تريكة `int(n * 0.8)`:** تحويل float → int (floor) — يضمن عدد صحيح.

**مثال:** `n = 45` → `split_idx = 36` → Train 36, Test 9.

**🎯 `return` metrics:**

```python
{
    "train_samples": float(len(X_train)),
    "test_samples": float(len(X_test)),
    "test_accuracy": round(self.last_accuracy * 100, 2),
}
```

**لماذا `float`؟** JSON serializable — `int` قد يسبب مشاكل في بعض الـ serializers.

---

#### `predict_proba` — الاحتمالية

```python
if not self.is_trained:
    raise ValidationError("...")
probs = self.model.predict_proba([features])[0]
return float(probs[1])
```

**🎯 `[features]`:** `predict_proba` يتوقع **2D array** — نلفّ المتجه المفرد في list.

**🎯 `[0]`:** إرجاع الصف الأول (العينة الوحيدة).

**🎯 `[1]`:** العمود الثاني = `P(y=1)` = **احتمالية الصعود**.

**مثال:** `probs = [0.35, 0.65]` → `return 0.65` (65% صعود).

---

## 📊 الفرق بين `predict` و `predict_proba`

| الدالة | المخرج | الاستخدام |
|--------|--------|-----------|
| `predict_proba` | `float [0,1]` | **درجة الثقة** |
| `predict` | `int {0,1}` | **القرار النهائي** |

**🎯 لماذا الاحتمالية مهمة؟**
- تتيح **حجم صفقة ديناميكي** (Sprint 8).
- تتيح **threshold مخصص** (مثلاً `> 0.65` فقط للدخول).
- تتيح **ensemble** مع نماذج أخرى.

---

## 🧪 التحقق (وصف)

**السيناريو:** 40 عينة، 5 features، y متناوب (0,1,0,1...).

| المقياس | المتوقع |
|---------|---------|
| Train Samples | 32 |
| Test Samples | 8 |
| Test Accuracy | ~50% (بيانات عشوائية) |
| Predict Proba | `[0, 1]` |
| Predict | 0 أو 1 |

**🎯 جوهر الاختبار:** إثبات أن النموذج **يتدرب**، **يتنبأ باحتمالية**، و**يصنف** بناءً عليها.

---

## ⚠️ ملاحظات على التصميم

| النقطة | التأثير |
|--------|---------|
| Split 80/20 ثابت | لا يدعم cross-validation |
| `int(n * 0.8)` | Test قد يكون صغيراً جداً لو n صغير |
| `random_state=42` | Training deterministic — **مطلوب** للاختبار |
| لا `StandardScaler` | الميزات مختلفة المقاييس → **قد يؤثر** |
| `min 20` ثابت | قد لا يكفي لنموذج 5 features |
| لا `TimeSeriesSplit` من sklearn | أبسط لكن أقل قوة |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| **Contract Enforcement** | 3 abstract methods |
| **DRY** | `predict` مشترك |
| **Time-Series Split** | `X[:split]` — لا shuffle |
| **Fail Early** | `n < 20`, `not is_trained` |
| **Probability Output** | `predict_proba` قبل `predict` |
| **Reproducibility** | `random_state=42` |
| **Serialization** | `float` in metrics |

---

## 🗺️ Sprint 7

```mermaid
graph LR
    A["ml/features.py ✅"] --> B["ml/models/base.py + logistic.py 🆕"]
    B --> C["services/forecast_service.py"]
    C --> D["CLI: forex forecast"]
    D --> E["tests + docs"]

    style B fill:#bfb,stroke:#333,stroke-width:3px
```

In [2]:
from ml.models.logistic import LogisticForecaster

# توليد بيانات ميزات وأهداف وهمية لاختبار عمل الكلاس
X_dummy = [[0.01 * (i % 5), -0.02 * (i % 3), 0.005, 0.55, 0.001] for i in range(40)]
y_dummy = [1 if i % 2 == 0 else 0 for i in range(40)]

forecaster = LogisticForecaster()
metrics = forecaster.train(X_dummy, y_dummy)

print('Model Name     :', forecaster.name)
print('Train Metrics  :', metrics)

# فحص تنبؤ عينة جديدة
sample_feature = [0.002, 0.01, 0.004, 0.65, 0.002]
up_prob = forecaster.predict_proba(sample_feature)
direction = 'UP 🟢' if forecaster.predict(sample_feature) == 1 else 'DOWN 🔴'

print(f'Upward Probability : {up_prob * 100:.2f}%')
print(f'Predicted Direction : {direction}')

Model Name     : LogisticRegression
Train Metrics  : {'train_samples': 32.0, 'test_samples': 8.0, 'test_accuracy': 50.0}
Upward Probability : 50.00%
Predicted Direction : UP 🟢


# 🔮 تحليل `ForecastService` — Sprint 7، الخطوة 3

---

## 🎯 TL;DR

> **`ForecastService`** يجمع `FeatureExtractor` + `LogisticForecaster` + `Storage`. **دالة واحدة**: `generate_forecast()` — تجلب الشموع، تستخرج الميزات، تُدرّب النموذج، تتنبأ، وتُرجع `ForecastReport` (11 حقل).

---

## 📊 المخطط المعماري

```mermaid
graph TD
    A[generate_forecast] --> B[CurrencyPair + Interval]
    B --> C[storage.get_candles]
    C --> D{len >= 60?}
    D -->|No| E[ValidationError + hint]
    D -->|Yes| F[sorted]
    F --> G[extractor.extract_features]
    G --> H{len X >= 20?}
    H -->|No| I[ValidationError]
    H -->|Yes| J[model.train]
    J --> K[extractor.extract_latest_features]
    K --> L{None?}
    L -->|Yes| M[ValidationError]
    L -->|No| N[model.predict_proba]
    N --> O[direction + probabilities]
    O --> P[ForecastReport]

    style J fill:#bfb,stroke:#333,stroke-width:2px
    style N fill:#bdf,stroke:#333,stroke-width:2px
    style P fill:#f9f,stroke:#333,stroke-width:2px
```

---

## 🔬 التشريح

### 1️⃣ `ForecastReport` — Value Object (11 حقل)

| الفئة | الحقول |
|-------|--------|
| **السياق** | `pair`, `interval`, `model_name` |
| **الشمعة** | `last_candle_time`, `latest_close` |
| **التنبؤ** | `direction`, `up_probability`, `down_probability` |
| **الأداء** | `test_accuracy`, `train_samples`, `test_samples` |

**🎯 `to_dict`:** تحويل لـ `float` + `round(2)` — للـ JSON/CLI.

**🎯 `down_probability = 100 - up_probability`:** ضمني — لا حساب منفصل.

---

### 2️⃣ `__init__` — DI ثلاثي

```python
self.storage = storage or SQLiteStorage()
self.extractor = extractor or FeatureExtractor()
self.model = model or LogisticForecaster()
```

**🎯 DI كامل:** يمكن حقن `MockExtractor` أو `DummyModel` في الاختبارات.

---

### 3️⃣ `generate_forecast` — 6 مراحل

#### المرحلة 1: التحقق + الجلب

```python
validated_pair = CurrencyPair.from_string(pair)
validated_interval = Interval.from_str(interval)
candles = self.storage.get_candles(..., limit=limit)
```

**Fail Early** — نفس نمط كل الخدمات.

#### المرحلة 2: فحص الحد الأدنى — **60 شمعة**

```python
min_candles_needed = 60
if len(candles) < min_candles_needed:
    raise ValidationError(
        f"... يلزم توفر {min_candles_needed} شمعة على الأقل.\n"
        f"استخدم أولاً: forex fetch --pair ... --limit {limit}"
    )
```

**🎯 لماذا 60؟**

| المصدر | الحد |
|--------|------|
| MACD(26) + Signal(9) | 34 شمعة |
| `FeatureExtractor` (min_required) | 35 شمعة |
| `LogisticForecaster` (min samples) | 20 عينة |
| **المجموع** | **35 + 20 = 55** |
| **الهامش الآمن** | **60** |

**🎯 تريكة Hint:** `ValidationError` يحتوي **الأمر المطلوب نسخه** — **UX ممتاز**.

#### المرحلة 3: استخراج الميزات

```python
X, y = self.extractor.extract_features(candles_sorted)
if len(X) < 20:
    raise ValidationError("تعذر استخراج عينات كافية...")
```

**🎯 لماذا فحص `len(X) < 20` منفصل؟**
- `extract_features` قد يرجع أقل من المتوقع (لو قيم `None` كثيرة).
- `LogisticForecaster.train` يرفض `< 20` — **فحص مسبق** لرسالة أوضح.

#### المرحلة 4: التدريب

```python
metrics = self.model.train(X, y)
```

**`metrics`** = `{train_samples, test_samples, test_accuracy}`.

**🎯 التدريب في كل استدعاء:** بسيط لكنه **مكلف** — التحسين في Sprint 8 (joblib caching).

#### المرحلة 5: ميزات الشمعة الأخيرة

```python
latest_features = self.extractor.extract_latest_features(candles_sorted)
if latest_features is None:
    raise ValidationError(...)
```

**🎯 `None` check:** لو المؤشرات ناقصة في الشمعة الأخيرة.

#### المرحلة 6: التنبؤ + التقرير

```python
up_prob = self.model.predict_proba(latest_features) * 100.0
down_prob = 100.0 - up_prob
direction = "UP" if up_prob >= 50.0 else "DOWN"
```

| المخرج | القيمة |
|--------|--------|
| `up_prob` | `0-100` (نسبة مئوية) |
| `down_prob` | `100 - up_prob` |
| `direction` | `"UP"` أو `"DOWN"` |

**🎯 `>= 50.0`:** متسق مع `predict()` في `BaseForecaster`.

---

## 📊 تدفق البيانات الكامل

```mermaid
sequenceDiagram
    participant CLI
    participant S as ForecastService
    participant DB as SQLite
    participant F as FeatureExtractor
    participant M as LogisticForecaster

    CLI->>S: generate_forecast("EUR/USD", "1day")
    S->>DB: get_candles(limit=150)
    DB-->>S: list[Candle]
    S->>S: sorted + min 60 check
    S->>F: extract_features(candles)
    F-->>S: X, y
    S->>M: train(X, y)
    M-->>S: metrics
    S->>F: extract_latest_features(candles)
    F-->>S: latest_features
    S->>M: predict_proba(latest_features)
    M-->>S: up_prob
    S-->>CLI: ForecastReport
```

---

## 🧪 التحقق (وصف)

| # | السيناريو | المتوقع |
|---|-----------|---------|
| 1 | 150 شمعة | تقرير كامل |
| 2 | 59 شمعة | `ValidationError` + hint |
| 3 | 70 شمعة | يعمل (هامش كافٍ) |
| 4 | 65 شمعة | يعمل لكن `test_samples` صغير |
| 5 | Model `random_state=42` | نتائج **deterministic** |

---

## ⚠️ ملاحظات على التصميم

| النقطة | التأثير |
|--------|---------|
| التدريب في كل استدعاء | بطيء (~1s) — يمكن cache |
| لا `joblib` caching | إعادة تدريب عند كل `forex forecast` |
| `limit=150` default | يستهلك DB كبير — لكن OK |
| `test_accuracy` فقط | لا precision/recall/F1 |
| لا confidence interval | `up_prob` نقطة واحدة |
| `direction = UP if >= 50` | متسق مع `predict` لكن مكرر |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| **Orchestration** | يجمع Storage + Extractor + Model |
| **DI ثلاثي** | Mock في الاختبارات |
| **Fail Early** | `min 60`, `len X >= 20` |
| **UX Hint** | "استخدم `forex fetch`..." |
| **Consistency** | `direction` = `predict()` |
| **Report Object** | 11 حقل موحد |
| **Serialization** | `to_dict` |

---

## 🗺️ Sprint 7

```mermaid
graph LR
    A["ml/features.py ✅"] --> B["ml/models/base.py + logistic.py ✅"]
    B --> C["services/forecast_service.py 🆕"]
    C --> D["CLI: forex forecast"]
    D --> E["tests + docs"]

    style C fill:#bfb,stroke:#333,stroke-width:3px
```

In [2]:
import math
from datetime import datetime, timedelta
from decimal import Decimal
from core.candle import Candle
from services.forecast_service import ForecastService
from storage.sqlite_storage import SQLiteStorage

storage = SQLiteStorage(db_path=':memory:')

# توليد 80 شمعة متذبذبة (صعود وهبوط)
start = datetime.now() - timedelta(days=80)
candles = []
for i in range(80):
    # إضافة حركة موجية تحاكي تذبذب السوق الحقيقي
    delta = math.sin(i * 0.3) * 0.0050
    base_price = 1.0800 + delta
    
    candles.append(
        Candle(
            pair='EUR/USD',
            interval='1day',
            timestamp=start + timedelta(days=i),
            open=Decimal(str(round(base_price - 0.0005, 5))),
            high=Decimal(str(round(base_price + 0.0015, 5))),
            low=Decimal(str(round(base_price - 0.0015, 5))),
            close=Decimal(str(round(base_price, 5))),
        )
    )

storage.save_candles_bulk(candles)

service = ForecastService(storage=storage)
report = service.generate_forecast(pair='EUR/USD', interval='1day', limit=80)

print('--- Forecast Report ---')
print('Pair           :', report.pair)
print('Model          :', report.model_name)
print('Latest Close   :', report.latest_close)
print('Direction      :', report.direction)
print('P(Up)          :', f'{report.up_probability:.1f}%')
print('P(Down)        :', f'{report.down_probability:.1f}%')
print('Test Accuracy  :', f'{report.test_accuracy:.1f}%')
print('Train Samples  :', report.train_samples)
print('Test Samples   :', report.test_samples)

[2026-11-06 23:22:33] [INFO] [forex_tracker]: Connected to SQLite database at: :memory:
[2026-11-06 23:22:33] [INFO] [forex_tracker]: Saved 80 candle(s) to SQLite.
[2026-11-06 23:22:33] [INFO] [forex_tracker]: Forecast generated for EUR/USD [1day]: Direction=DOWN, P(Up)=49.8%, TestAcc=20.0%


--- Forecast Report ---
Pair           : EUR/USD
Model          : LogisticRegression
Latest Close   : 1.07505
Direction      : DOWN
P(Up)          : 49.8%
P(Down)        : 50.2%
Test Accuracy  : 20.0%
Train Samples  : 39
Test Samples   : 10
